# GELU checker cost on a GPU

Tests the three open risks of the proposal on one GPU (Colab T4 is enough):

1. **Speed.** Does the Route B checker cost less than recomputing GELU? Measured three ways:
   as a separate checker kernel, fused into a standalone GELU kernel, and fused into a matrix-multiply epilogue.
2. **Clean false alarms.** Real GELU kernels are not correctly rounded. How often does Route B flag a
   correct output from a production kernel?
3. **Impact.** What share of GPU time does GELU take in real models?

Runtime: about 5–10 minutes on a T4. Runtime → Change runtime type → T4 GPU, then Run all.

In [ ]:
import math
import subprocess
import sys

import numpy as np
import torch
import triton
import triton.language as tl

try:                                             # libdevice moved between Triton 3.x releases
    from triton.language.extra import libdevice
except ImportError:
    from triton.language.extra.cuda import libdevice

assert torch.cuda.is_available(), "Select a GPU runtime (Runtime -> Change runtime type -> T4 GPU)"
print("torch", torch.__version__, "| triton", triton.__version__, "|", torch.cuda.get_device_name())

# Reuse the CPU reference package from the project repo (table construction, exact GELU).
subprocess.run(["git", "clone", "-q", "https://github.com/SeanoChang/ece695-system-ai-project", "/content/repo"])
sys.path.insert(0, "/content/repo")
from gelucheck import route_b
from gelucheck.formats import fp16_cell
from gelucheck.gelu import gelu as gelu64

DEV = "cuda"
TAB = route_b.build(-5.0, 5.0, 1 / 16)
COEF = torch.tensor(TAB.coef32, device=DEV).contiguous()        # (160, 4) float32
DELTA = torch.tensor(TAB.delta32, device=DEV).contiguous()      # (160,)  float32
LO, H, NT = float(TAB.lo), float(TAB.h), int(TAB.n)
print(f"table: {NT} intervals, {TAB.nbytes} bytes")

## Shared device functions

`_gelu` is the production kernel's formula. `_gelu2` is the duplicate: it uses `erfc` (a different libdevice
routine), so the compiler cannot merge it with the first `erf` call. Section 1 checks this in the PTX. `_certify` is Route B: interval lookup, cubic, and the rounding-cell
test. The cell test uses a margin of one FP32 ulp of `p` so that FP32 rounding in the test itself can never
certify a wrong value.

In [ ]:
@triton.jit
def _gelu(x):
    return 0.5 * x * (1.0 + tl.erf(x * 0.7071067811865476))


@triton.jit
def _gelu2(x):
    # erfc(-z) = 1 + erf(z), computed by a different libdevice routine than erf
    return 0.5 * x * libdevice.erfc(-x * 0.7071067811865476)


@triton.jit
def _bits16(v16):
    return v16.to(tl.int16, bitcast=True).to(tl.int32) & 0xFFFF


@triton.jit
def _f16_from_bits(b):
    return (b & 0xFFFF).to(tl.int16).to(tl.float16, bitcast=True).to(tl.float32)


@triton.jit
def _certify(x, coef_ptr, delta_ptr, LO: tl.constexpr, H: tl.constexpr, NT: tl.constexpr):
    jf = tl.floor((x - LO) * (1.0 / H))
    j = tl.minimum(tl.maximum(jf, 0.0), NT - 1.0).to(tl.int32)
    t = x - (LO + j.to(tl.float32) * H)
    c0 = tl.load(coef_ptr + j * 4 + 0)
    c1 = tl.load(coef_ptr + j * 4 + 1)
    c2 = tl.load(coef_ptr + j * 4 + 2)
    c3 = tl.load(coef_ptr + j * 4 + 3)
    d = tl.load(delta_ptr + j)
    p = ((c3 * t + c2) * t + c1) * t + c0
    vb = _bits16(p.to(tl.float16))
    vf = _f16_from_bits(vb)
    n1 = _f16_from_bits(vb - 1)
    n2 = _f16_from_bits(vb + 1)
    mlo = (vf + tl.minimum(n1, n2)) * 0.5
    mhi = (vf + tl.maximum(n1, n2)) * 0.5
    margin = d + tl.abs(p) * 1.2e-7
    inr = (x >= LO) & (x < LO + NT * H)
    ok = inr & ((p - mlo) > margin) & ((mhi - p) > margin) & ((vb & 0x7FFF) != 0)
    return ok, vb


@triton.jit
def _append(unc, idx, val, cnt_ptr, list_ptr, xval_ptr):
    """Append (index, FP32 GELU input) where unc is true to global lists (one atomic per program)."""
    u = unc.to(tl.int32)
    base = tl.atomic_add(cnt_ptr, tl.sum(u, axis=0))
    pos = base + tl.cumsum(u, axis=0) - 1
    tl.store(list_ptr + pos, idx, mask=unc)
    tl.store(xval_ptr + pos, val, mask=unc)

## Standalone kernels (FP32 in, FP16 out)

| Kernel | What it does |
|---|---|
| `gelu` | unprotected baseline |
| `gelu_dup` | GELU plus a duplicate, compared bit for bit |
| `gelu_range` | GELU plus a range check: `max(x, −0.1701) − tol ≤ y ≤ max(x, 0) + tol` |
| `gelu_rb_block` | GELU plus Route B; if any input in the block is uncertified, recompute the whole block |
| `gelu_rb_list` | GELU plus Route B; uncertified inputs go to a list for a second, small kernel |
| `check_dup` / `check_rb` | separate checker kernels that read `x` and `y` after the GELU kernel |

In [ ]:
@triton.jit
def k_gelu(x_ptr, y_ptr, n, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    m = offs < n
    x = tl.load(x_ptr + offs, mask=m, other=1.0)
    tl.store(y_ptr + offs, _gelu(x).to(tl.float16), mask=m)


@triton.jit
def k_gelu_dup(x_ptr, y_ptr, bad_ptr, n, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    m = offs < n
    x = tl.load(x_ptr + offs, mask=m, other=1.0)
    y = _gelu(x).to(tl.float16)
    tl.store(y_ptr + offs, y, mask=m)
    bad = m & (_bits16(_gelu2(x).to(tl.float16)) != _bits16(y))
    tl.atomic_add(bad_ptr, tl.sum(bad.to(tl.int32), axis=0))


@triton.jit
def k_gelu_range(x_ptr, y_ptr, bad_ptr, n, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    m = offs < n
    x = tl.load(x_ptr + offs, mask=m, other=1.0)
    y = _gelu(x).to(tl.float16)
    tl.store(y_ptr + offs, y, mask=m)
    yf = y.to(tl.float32)
    tol = 1e-3 * tl.abs(x) + 1e-4
    bad = m & ((yf < tl.maximum(x, -0.1701) - tol) | (yf > tl.maximum(x, 0.0) + tol))
    tl.atomic_add(bad_ptr, tl.sum(bad.to(tl.int32), axis=0))


@triton.jit
def k_gelu_rb_block(x_ptr, y_ptr, bad_ptr, fb_ptr, coef_ptr, delta_ptr, n,
                    LO: tl.constexpr, H: tl.constexpr, NT: tl.constexpr, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    m = offs < n
    x = tl.load(x_ptr + offs, mask=m, other=1.0)
    y = _gelu(x).to(tl.float16)
    tl.store(y_ptr + offs, y, mask=m)
    yb = _bits16(y)
    ok, vb = _certify(x, coef_ptr, delta_ptr, LO, H, NT)
    ok = ok | (offs >= n)
    bad = m & ok & (vb != yb)
    unc = m & (ok.to(tl.int32) == 0)
    if tl.sum(unc.to(tl.int32), axis=0) > 0:
        bad = bad | (unc & (_bits16(_gelu2(x).to(tl.float16)) != yb))
        tl.atomic_add(fb_ptr, 1)
    tl.atomic_add(bad_ptr, tl.sum(bad.to(tl.int32), axis=0))


@triton.jit
def k_gelu_rb_list(x_ptr, y_ptr, bad_ptr, cnt_ptr, list_ptr, xval_ptr, coef_ptr, delta_ptr, n,
                   LO: tl.constexpr, H: tl.constexpr, NT: tl.constexpr, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    m = offs < n
    x = tl.load(x_ptr + offs, mask=m, other=1.0)
    y = _gelu(x).to(tl.float16)
    tl.store(y_ptr + offs, y, mask=m)
    ok, vb = _certify(x, coef_ptr, delta_ptr, LO, H, NT)
    ok = ok | (offs >= n)
    bad = m & ok & (vb != _bits16(y))
    tl.atomic_add(bad_ptr, tl.sum(bad.to(tl.int32), axis=0))
    _append(m & (ok.to(tl.int32) == 0), offs, x, cnt_ptr, list_ptr, xval_ptr)


@triton.jit
def k_check_dup(x_ptr, y_ptr, bad_ptr, n, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    m = offs < n
    x = tl.load(x_ptr + offs, mask=m, other=1.0)
    y = tl.load(y_ptr + offs, mask=m, other=0.0)
    bad = m & (_bits16(_gelu2(x).to(tl.float16)) != _bits16(y))
    tl.atomic_add(bad_ptr, tl.sum(bad.to(tl.int32), axis=0))


@triton.jit
def k_check_rb(x_ptr, y_ptr, bad_ptr, cnt_ptr, list_ptr, xval_ptr, coef_ptr, delta_ptr, n,
               LO: tl.constexpr, H: tl.constexpr, NT: tl.constexpr, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    m = offs < n
    x = tl.load(x_ptr + offs, mask=m, other=1.0)
    y = tl.load(y_ptr + offs, mask=m, other=0.0)
    ok, vb = _certify(x, coef_ptr, delta_ptr, LO, H, NT)
    ok = ok | (offs >= n)
    bad = m & ok & (vb != _bits16(y))
    tl.atomic_add(bad_ptr, tl.sum(bad.to(tl.int32), axis=0))
    _append(m & (ok.to(tl.int32) == 0), offs, x, cnt_ptr, list_ptr, xval_ptr)


@triton.jit
def k_fallback(xval_ptr, list_ptr, cnt_ptr, y_ptr, bad_ptr, BLOCK: tl.constexpr):
    """Second pass: recompute only the uncertified inputs and compare.

    Reads the count on the device (no host sync) and walks the list with a fixed grid.
    """
    count = tl.load(cnt_ptr)
    step = tl.num_programs(0) * BLOCK
    for start in range(tl.program_id(0) * BLOCK, count, step):
        k = start + tl.arange(0, BLOCK)
        m = k < count
        i = tl.load(list_ptr + k, mask=m, other=0)
        x = tl.load(xval_ptr + k, mask=m, other=1.0)
        y = tl.load(y_ptr + i, mask=m, other=0.0)
        bad = m & (_bits16(_gelu2(x).to(tl.float16)) != _bits16(y))
        tl.atomic_add(bad_ptr, tl.sum(bad.to(tl.int32), axis=0))

In [ ]:
BLOCK = 1024
CONST = dict(LO=LO, H=H, NT=NT)


def zeros1():
    return torch.zeros(1, dtype=torch.int32, device=DEV)


def run_variant(name, x, y, bad, aux):
    n = x.numel()
    g = (triton.cdiv(n, BLOCK),)
    if name == "gelu":
        k_gelu[g](x, y, n, BLOCK=BLOCK)
    elif name == "gelu_dup":
        k_gelu_dup[g](x, y, bad, n, BLOCK=BLOCK)
    elif name == "gelu_range":
        k_gelu_range[g](x, y, bad, n, BLOCK=BLOCK)
    elif name == "gelu_rb_block":
        k_gelu_rb_block[g](x, y, bad, aux["fb"], COEF, DELTA, n, BLOCK=BLOCK, **CONST)
    elif name == "gelu_rb_list":
        aux["cnt"].zero_()
        k_gelu_rb_list[g](x, y, bad, aux["cnt"], aux["list"], aux["xval"], COEF, DELTA, n, BLOCK=BLOCK, **CONST)
        fallback(y, bad, aux)
    elif name == "check_dup":
        k_check_dup[g](x, y, bad, n, BLOCK=BLOCK)
    elif name == "check_rb":
        aux["cnt"].zero_()
        k_check_rb[g](x, y, bad, aux["cnt"], aux["list"], aux["xval"], COEF, DELTA, n, BLOCK=BLOCK, **CONST)
        fallback(y, bad, aux)
    else:
        raise ValueError(name)


FB_GRID = 40   # fixed grid for the second pass: a few programs per SM, no host sync


def fallback(y, bad, aux):
    k_fallback[(FB_GRID,)](aux["xval"], aux["list"], aux["cnt"], y, bad, BLOCK=BLOCK)


def new_aux(n):
    return {"fb": zeros1(), "cnt": zeros1(), "list": torch.empty(n, dtype=torch.int32, device=DEV),
            "xval": torch.empty(n, dtype=torch.float32, device=DEV)}

## 1. Correctness on clean data, and clean false alarms

Inputs are standard normal (FP32). The reference is double-precision GELU, correctly rounded to FP16.

In [ ]:
torch.manual_seed(0)
N_CHECK = 1 << 22
x = torch.randn(N_CHECK, device=DEV)
y = torch.empty(N_CHECK, dtype=torch.float16, device=DEV)
aux = new_aux(N_CHECK)

k_gelu[(triton.cdiv(N_CHECK, BLOCK),)](x, y, N_CHECK, BLOCK=BLOCK)
xs = x.cpu().numpy().astype(np.float64)
ref_bits, _, _ = fp16_cell(gelu64(xs))
kern_bits = y.cpu().numpy().view(np.uint16)
not_cr = int((ref_bits != kern_bits).sum())
print(f"production kernel outputs that are not correctly rounded: {not_cr} of {N_CHECK} ({not_cr / N_CHECK:.2e})")

for name in ("gelu_dup", "gelu_range", "gelu_rb_block", "gelu_rb_list"):
    bad = zeros1()
    aux["fb"].zero_()
    run_variant(name, x, y, bad, aux)
    extra = ""
    if name == "gelu_rb_list":
        c = int(aux["cnt"].item())
        extra = f"; uncertified {c} ({100 * c / N_CHECK:.3f}%) -> second pass"
    if name == "gelu_rb_block":
        nb = triton.cdiv(N_CHECK, BLOCK)
        extra = f"; blocks needing full recompute {int(aux['fb'].item())} of {nb}"
    print(f"{name:14s} false alarms on clean data: {int(bad.item())}{extra}")

In [ ]:
# Does the compiler really compute GELU twice? Compare PTX sizes (the duplicate should be clearly larger).
_g = (triton.cdiv(N_CHECK, BLOCK),)
_b = zeros1()
ptx_one = k_gelu[_g](x, y, N_CHECK, BLOCK=BLOCK).asm["ptx"]
ptx_dup = k_gelu_dup[_g](x, y, _b, N_CHECK, BLOCK=BLOCK).asm["ptx"]
count = lambda p: sum(1 for l in p.splitlines() if l.strip() and not l.strip().startswith(("//", ".", "{", "}", "$")))
print(f"PTX instructions: gelu {count(ptx_one)}, gelu_dup {count(ptx_dup)}")

## 2. Fault injection: flip one random bit in K outputs, then run the separate checkers

In [ ]:
K = 2000
gen = torch.Generator(device="cpu").manual_seed(1)
pos = torch.randperm(N_CHECK, generator=gen)[:K]
bit = torch.randint(0, 16, (K,), generator=gen)
yf = y.clone()
yv = yf.view(torch.int16).cpu().numpy()
yv[pos.numpy()] ^= (np.uint16(1) << bit.numpy().astype(np.uint16)).astype(np.int16)
yf = torch.tensor(yv, device=DEV).view(torch.float16)

for name in ("check_dup", "check_rb"):
    base, inj = zeros1(), zeros1()
    run_variant(name, x, y, base, aux)
    run_variant(name, x, yf, inj, aux)
    print(f"{name:10s} flagged on clean y: {int(base.item())}; on y with {K} flipped outputs: {int(inj.item())} "
          f"(expect about {K} more than clean; a flip on an already-flagged lane can cancel)")

## 3. Timing: standalone kernels

Sizes: decode-like (16 tokens), prefill-like (512 and 4096 tokens), each times the 4096-wide MLP of Pythia-410M.
`rb_list` and `check_rb` timings include the second pass and the reset of the list counter (a few us).

In [ ]:
def bench(fn):
    return triton.testing.do_bench(fn, warmup=25, rep=200)


rows = []
for tokens in (16, 512, 4096):
    n = tokens * 4096
    x = torch.randn(n, device=DEV)
    y = torch.empty(n, dtype=torch.float16, device=DEV)
    bad = zeros1()
    aux = new_aux(n)
    k_gelu[(triton.cdiv(n, BLOCK),)](x, y, n, BLOCK=BLOCK)
    t = {}
    for name in ("gelu", "gelu_dup", "gelu_range", "gelu_rb_block", "gelu_rb_list", "check_dup", "check_rb"):
        t[name] = bench(lambda: run_variant(name, x, y, bad, aux))
    rows.append((tokens, n, t))
    gbs = 6 * n / (t["gelu"] * 1e-3) / 1e9
    print(f"\n{tokens} tokens ({n:,} elements); unprotected GELU {t['gelu'] * 1e3:.1f} us = {gbs:.0f} GB/s")
    for name, v in t.items():
        print(f"  {name:14s} {v * 1e3:9.1f} us   x{v / t['gelu']:.2f} of unprotected")

## 4. Timing: GELU fused into the matrix-multiply epilogue

`C = GELU(A @ B)` with A (M x 1024) and B (1024 x 4096) in FP16 and an FP32 accumulator, as in the Pythia-410M MLP.
Modes: 0 no activation, 1 GELU, 2 GELU + duplicate, 3 GELU + Route B with tile-level fallback,
4 GELU + Route B with the uncertified list.

In [ ]:
@triton.jit
def k_mm(a_ptr, b_ptr, c_ptr, bad_ptr, cnt_ptr, list_ptr, xval_ptr, coef_ptr, delta_ptr, M, N, K,
         MODE: tl.constexpr, LO: tl.constexpr, H: tl.constexpr, NT: tl.constexpr,
         BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pm = tl.program_id(0)
    pn = tl.program_id(1)
    rm = pm * BM + tl.arange(0, BM)
    rn = pn * BN + tl.arange(0, BN)
    rk = tl.arange(0, BK)
    acc = tl.zeros((BM, BN), dtype=tl.float32)
    for k0 in range(0, K, BK):
        a = tl.load(a_ptr + rm[:, None] * K + (k0 + rk)[None, :],
                    mask=(rm[:, None] < M) & ((k0 + rk)[None, :] < K), other=0.0)
        b = tl.load(b_ptr + (k0 + rk)[:, None] * N + rn[None, :],
                    mask=((k0 + rk)[:, None] < K) & (rn[None, :] < N), other=0.0)
        acc += tl.dot(a, b)
    m = (rm[:, None] < M) & (rn[None, :] < N)
    cptr = c_ptr + rm[:, None] * N + rn[None, :]
    if MODE == 0:
        tl.store(cptr, acc.to(tl.float16), mask=m)
    else:
        y = _gelu(acc).to(tl.float16)
        tl.store(cptr, y, mask=m)
        yb = _bits16(y)
        if MODE == 2:
            bad = m & (_bits16(_gelu2(acc).to(tl.float16)) != yb)
            tl.atomic_add(bad_ptr, tl.sum(tl.sum(bad.to(tl.int32), axis=1), axis=0))
        if MODE >= 3:
            ok, vb = _certify(acc, coef_ptr, delta_ptr, LO, H, NT)
            ok = ok | (rm[:, None] >= M) | (rn[None, :] >= N)
            bad = m & ok & (vb != yb)
            unc = m & (ok.to(tl.int32) == 0)
            if MODE == 3:
                if tl.sum(tl.sum(unc.to(tl.int32), axis=1), axis=0) > 0:
                    bad = bad | (unc & (_bits16(_gelu2(acc).to(tl.float16)) != yb))
            if MODE == 4:
                idx = rm[:, None] * N + rn[None, :]
                _append(tl.reshape(unc, (BM * BN,)), tl.reshape(idx, (BM * BN,)), tl.reshape(acc, (BM * BN,)),
                        cnt_ptr, list_ptr, xval_ptr)
            tl.atomic_add(bad_ptr, tl.sum(tl.sum(bad.to(tl.int32), axis=1), axis=0))


def mm(a, b, c, mode, bad, aux):
    M, K = a.shape
    N = b.shape[1]
    BM = 16 if M <= 16 else 64
    if mode == 4:
        aux["cnt"].zero_()
    k_mm[(triton.cdiv(M, BM), triton.cdiv(N, 64))](a, b, c, bad, aux["cnt"], aux["list"], aux["xval"], COEF, DELTA,
                                                   M, N, K, MODE=mode, BM=BM, BN=64, BK=32, num_warps=4,
                                                   num_stages=2, **CONST)
    if mode == 4:
        fallback(c.reshape(-1), bad, aux)


K_DIM, N_DIM = 1024, 4096
for M in (16, 512, 2048):
    a = (torch.randn(M, K_DIM, device=DEV) / math.sqrt(K_DIM)).half()   # keeps GELU inputs near N(0, 1)
    b = torch.randn(K_DIM, N_DIM, device=DEV).half()
    c = torch.empty(M, N_DIM, dtype=torch.float16, device=DEV)
    bad = zeros1()
    aux = new_aux(M * N_DIM)
    times = {}
    for mode, label in ((0, "matmul only"), (1, "+ GELU"), (2, "+ GELU + duplicate"),
                        (3, "+ GELU + Route B (tile fallback)"), (4, "+ GELU + Route B (list)")):
        try:
            times[label] = bench(lambda: mm(a, b, c, mode, bad, aux))
        except Exception as e:  # keep the notebook going if one variant fails to compile
            times[label] = float("nan")
            print(f"  mode {mode} failed: {type(e).__name__}: {str(e)[:200]}")
    base = times["+ GELU"]
    print(f"\nM = {M} tokens")
    for label, v in times.items():
        print(f"  {label:34s} {v * 1e3:9.1f} us   x{v / base:.3f} of matmul + GELU")

## 5. Impact: share of GPU time spent in GELU in real models (FP16, eager PyTorch)

In [ ]:
from torch.profiler import ProfilerActivity, profile
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer


def gelu_share(fn):
    """Share of GPU kernel time spent in kernels whose name contains 'gelu' (kernel events only)."""
    fn()
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CUDA]) as prof:
        fn()
        torch.cuda.synchronize()
    kern = [e for e in prof.events() if e.device_type == torch.autograd.DeviceType.CUDA]
    total = sum(e.time_range.elapsed_us() for e in kern)
    g = sum(e.time_range.elapsed_us() for e in kern if "gelu" in e.name.lower())
    if g == 0:
        print("  (no kernel named *gelu* found; names seen:", sorted({e.name[:60] for e in kern})[:8], ")")
    return g / total if total else float("nan")


with torch.no_grad():
    tok = AutoTokenizer.from_pretrained("EleutherAI/pythia-410m")
    lm = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-410m", torch_dtype=torch.float16).to(DEV).eval()
    ids = torch.randint(0, tok.vocab_size, (1, 512), device=DEV)
    print(f"Pythia-410M prefill (1 x 512 tokens): GELU {100 * gelu_share(lambda: lm(ids)):.2f}% of GPU time")
    print(f"Pythia-410M decode (64 new tokens):   GELU "
          f"{100 * gelu_share(lambda: lm.generate(ids[:, :32], max_new_tokens=64, do_sample=False)):.2f}% of GPU time")
    del lm
    bert = AutoModel.from_pretrained("google-bert/bert-base-uncased", torch_dtype=torch.float16).to(DEV).eval()
    bids = torch.randint(0, 30000, (32, 128), device=DEV)
    print(f"BERT-base (32 x 128 tokens):          GELU {100 * gelu_share(lambda: bert(bids)):.2f}% of GPU time")